# Lab 5\. Reading a DistributedDataParallel Setup Script

## Overview

In the chapter, you learned the one idea that data-parallel training rests on: put a full copy of the model on each device, split the data so each copy trains on a different batch, and average the gradients every step so the copies remain identical. You also learned how PyTorch turns that idea into working code with DistributedDataParallel (DDP), and you met the four terms that appear in almost any distributed script — rank, world size, process group, and backend. This lab is where you put a name to each part of a real script.

You will work through a complete, short DDP training script that runs across CPU processes on a single machine, so you need no GPU and no cluster. Rather than write the script from scratch, you will read it the way you will most often meet distributed code in practice — in a repository, a tutorial, or a code review — and annotate it section by section. For each section you will say what it does, predict what would break if it were removed, and match it to a term from the chapter. An optional final section lets you run the script yourself if you want to see the processes start.

Being able to read a DDP script with intent is the practical skill this chapter builds toward. Distributed training code looks like a lot of boilerplate until you can point at each piece and explain its job. Once you can, an unfamiliar training script stops being a wall of setup and becomes four or five recognizable parts you already understand.

## Learning Objectives

By the end of this lab, you should be able to:

- Identify the five structural pieces of a DDP script: process-group initialization, the DDP model wrapper, the `DistributedSampler`, saving to disk, and process-group teardown  
- Explain the role of `torch.distributed.init_process_group` and `DistributedSampler` in a running script  
- Connect each section of a DDP script to the terms rank, world size, process group, and backend  
- Predict the consequence of removing a given line, such as the sampler or the cleanup call  
- Recognize where gradient synchronization happens in the training loop without any explicit synchronization code

## Prerequisites

This course’s labs are **not cumulative**, so this lab does not depend on any prior resources or the state of earlier labs if there are any.  

You need the following before you start:

- **A notebook environment:** either a Google account for Google Colab, or a local installation of Jupyter Notebook. Every step in this lab runs on a CPU; no GPU is required.  
- **Python 3.12 or later.** On Colab this is already provided. For a local environment, confirm your version with `python --version`.  
- **PyTorch 2.12.1,** installed with `pip` in a previous lab. If you are on Colab, these may already be present.  
- **A web browser** (Chrome, Firefox, or Edge are recommended for Colab).  
- **To read and annotate:** any code editor or a notebook cell where you can view the script. You will not need to execute anything for the core exercise.

> **Note:** No GPU is required. The lab detects your hardware and runs on CPU by default. If you happen to have a CUDA GPU available (for example, by enabling the Colab GPU runtime), the same code will use it, and the timing comparison near the end will be more interesting.

## Estimated Time

Approximately 25–35 minutes for the reading and annotation. Add about 10 minutes if you complete the optional run.

## How to Perform the Exercise

You will work entirely in one notebook, running cells from top to bottom. Create a new notebook in Google Colab (**File \> New notebook**) or a new `.ipynb` in local Jupyter. Download the chapter 5 lab notebook from the [LFD2003-resources GitHub repository](https://github.com/lftraining/LFD2003-resources.git) from the **LFD2003-resources/labs** directory. In Google Colab, select File, then Upload Notebook, and select the chapter 5 lab notebook you downloaded.

Read the explanation before running each cell, run it, and compare what you see to the **Sample Output** or **Output** shown after each block. Sample Output means the exact values will differ from run to run (random tensors, dataset counts, file paths), so match the shape and structure rather than every character. Output marks a checkpoint where a specific outcome confirms the step worked.

Commands in this lab use the notebook form, where a line beginning with `!` runs a shell command from inside a notebook cell (for example, `!pip install ...`). If you run these in a local terminal instead of a notebook cell, drop the leading `!`.

> **Note:** Run the cells in order and only once each, unless a step says otherwise. If you restart the notebook's runtime, re-run the cells from the top, because the variables live in memory and are cleared on restart.

## The Complete Script

Read this through once before annotating. It trains a small linear regression model on synthetic data across two CPU processes. The feature data is random tensors so the script is self-contained and needs no download, and the targets are created using a few predetermined weights and some random noise.


In [ ]:
import copy
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, TensorDataset, DistributedSampler


def make_model(seed=0):
    torch.manual_seed(seed)
    return nn.Linear(4, 1)

def make_dataset(n_samples=512):
    # Creates four features of random values
    X = torch.randn(n_samples, 4)

    # Synthetic regression target
    true_w = torch.tensor([[2.0], [-1.0], [0.5], [3.0]])
    y = X @ true_w + 0.1 * torch.randn(n_samples, 1)
    # Include sample indices so we can show which rank saw which examples
    indices = torch.arange(n_samples)

    dataset = TensorDataset(indices, X, y)
    return dataset


def setup(rank, world_size):
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12355"
    dist.init_process_group("gloo", rank=rank, world_size=world_size)


def save_model(model, filepath):
    torch.save(model.state_dict(), filepath)

def cleanup():
    dist.destroy_process_group()


def train(rank, world_size, initial_model, dataset, epochs=3):
    setup(rank, world_size)

    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    loader = DataLoader(dataset, batch_size=32, sampler=sampler, shuffle=False)


    model = copy.deepcopy(initial_model)
    model = DDP(model)

    loss_fn = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1)

    for epoch in range(epochs):
        sampler.set_epoch(epoch) # so each epoch gets a different distributed shuffle
        epoch_loss = 0.0
        seen_indices = []

        for indices, features, labels in loader:
            optimizer.zero_grad()
            outputs = model(features)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            seen_indices.extend(indices.tolist())

        print(
            f"\nrank={rank} | epoch={epoch} | "
            f"loss={epoch_loss:.4f} | "
            f"saw indices={seen_indices}"
        )

    # Make sure all ranks have finished training before rank 0 saves.
    dist.barrier()

    if rank == 0:
        save_model(model, "ddp_trained_model.pt")


    cleanup()


def main():
    initial_model = make_model()
    dataset = make_dataset()

    world_size = 2
    num_epochs=10
    mp.start_processes(train, args=(world_size, initial_model, dataset, num_epochs), nprocs=world_size, start_method='fork', join=True)


if __name__ == "__main__":
    main()


## Annotating the Script

Work through the sections in order. For each one, read the explanation, answer the annotation questions in your own words, then check the answer block. The questions are the same three each time: what the section does, what would happen if you removed it, and which chapter term it corresponds to.

### Step 1\. The Data

This first block is ordinary single-device PyTorch and is here so the script is complete and runnable. `make_model` is a small linear model that takes four features and produces a single numerical output. `make_dataset` builds a `TensorDataset` of 512 samples of four features each and 512 targets obtained by using predetermined weights and some added random noise , standing in for a real dataset so the script needs no download.

The point to notice is what is *not* here. Nothing in this block is distributed. The model definition and the dataset are exactly what you would write for one device; the distributed parts are layered on top later, without changing the model at all. That is the promise from the chapter made concrete: DDP wraps an ordinary model rather than replacing it.

**Annotate it:**

1. What does this block define, and is any of it specific to distributed training?  
2. What would happen to the rest of the script if `make_dataset` returned only 8 samples instead of 512?  
3. Which idea from the chapter does "the model is unchanged" correspond to?

**What you should have found:**

1. It defines the model (`make_model`) and a synthetic dataset (`make_dataset`). None of it is distributed; it is standard single-device PyTorch.  
2. With only 8 samples split across 2 processes and a batch size of 32, each process would receive a fraction of a single batch, so the training loop would still run but each process would see very little data per epoch. The script would not error, but the run would not be meaningful. This shows that the data volume, not the distributed code, is what fills the per-process batches.  
3. It matches the chapter's point that a model replica is a full, unchanged copy of the ordinary model; data parallelism does not require rewriting the model.

### Step 2\. Initializing the Process Group

The `setup` function is the entry point for everything distributed. A process group is the set of processes that will communicate with one another, and until it exists, a process cannot talk to the others at all.


In [ ]:
def setup(rank, world_size):
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12355"
    dist.init_process_group("gloo", rank=rank, world_size=world_size)


The two environment values tell the processes where to find each other: `MASTER_ADDR` is the address of the coordinating process, and `MASTER_PORT` is a free port they all agree on. Here both point at the local machine, because all the processes run on one computer. The call to `init_process_group` then creates the process group and does three things at once: it names the backend, and it records this process's `rank` and the total `world_size`.

The first argument, `"gloo"`, is the backend, which is the library that actually moves data between processes. Gloo is the backend that runs on CPU, which is why it is the right choice here; a job on NVIDIA GPUs would pass `"nccl"` instead. The `rank` is this process's unique identification number, and `world_size` is the total number of processes in the job.

**Annotate it:**

1. What does the line `dist.init_process_group("gloo", ...)` do, and why is the argument `"gloo"` rather than `"nccl"`?  
2. What would happen if you removed this call but kept the rest of the `train` function?  
3. Which four chapter terms appear in or are set by this function?

**What you should have found:**

1. It creates the process group and names the communication backend. `"gloo"` is used because the script runs on CPU, and Gloo is the CPU backend; `"nccl"` is for NVIDIA GPUs.  
2. Every later distributed operation would fail, because the model wrapper and the gradient averaging depend on an existing process group. Nothing distributed can run before this call succeeds.  
3. Process group (created here), backend (`"gloo"`), rank, and world size (both passed in and recorded).

### Step 3\. Splitting the Data with DistributedSampler

Inside `train`, the first distributed-specific lines build the data pipeline so that each process trains on a different slice of the dataset.


In [ ]:

    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    loader = DataLoader(dataset, batch_size=32, sampler=sampler, shuffle=False)


A `DistributedSampler` hands each process a different, non-overlapping subset of the dataset. It uses `num_replicas` (the world size) to know how many slices to make and `rank` to know which slice belongs to this process. You pass the sampler to the `DataLoader` through the `sampler` argument and set `shuffle=False`, because the sampler is now responsible for shuffling; setting `shuffle=True` on the loader at the same time is not allowed and would conflict with the sampler.

Note also that `batch_size=32` is the batch size *per process*. With a world size of 2, the two processes together handle 64 samples per step, which is the larger effective batch the chapter described.

**Annotate it:**

1. What does `DistributedSampler` do, and why is `shuffle=False` set on the `DataLoader`?  
2. What would happen if you removed the sampler and loaded the data the ordinary way (a plain `DataLoader` with `shuffle=True`)?  
3. Which chapter terms does the sampler rely on to decide what each process receives?

**What you should have found:**

1. It gives each process a different, non-overlapping slice of the dataset. `shuffle=False` is set because the sampler handles shuffling; asking the loader to shuffle as well conflicts with the sampler.  
2. Without the sampler, every process would read the same batches, so the two replicas would do duplicate work instead of dividing the data. The run would waste the second process rather than train faster.  
3. It relies on the world size (`num_replicas`) to know how many slices to cut and the rank to know which slice this process gets.

### Step 4\. Wrapping the Model in DDP

Two lines turn the ordinary model into a synchronized replica.


In [ ]:

    model = copy.deepcopy(initial_model)
    model = DDP(model)


After `DDP(model)`, you train the `model` exactly as you would a normal model. The wrapper adds gradient synchronization behind the backward pass, so the averaging happens with no extra code in the training loop. On a GPU job you would also pass `device_ids` to tell DDP which device this process owns; on a CPU job, as here, you leave that out and DDP works on the CPU model directly.

> **Note:** The order matters. The process group must already exist (from Step 2\) before you wrap the model, because DDP uses that group to synchronize the replicas the moment it is constructed.

**Annotate it:**

1. What does wrapping the model in `DDP` add that a plain model does not have?  
2. What would happen if you called `DDP(model)` before `init_process_group` had run?  
3. Why does this CPU script omit the `device_ids` argument that GPU examples include?

**What you should have found:**

1. It adds gradient synchronization: during the backward pass, DDP averages each parameter's gradient across all processes so the replicas stay identical.  
2. It would fail, because DDP relies on an existing process group to coordinate the replicas; the group must be created first.  
3. `device_ids` names a specific GPU for the process to use. On CPU there is no GPU device to assign, so the argument is left out and DDP wraps the CPU model directly.

### Step 5\. The Training Loop and Where Gradients Synchronize

The loop looks almost exactly like a single-device training loop, which is the point.


In [ ]:
    for epoch in range(epochs):
        sampler.set_epoch(epoch)
        epoch_loss = 0.0
        seen_indices = []

        for features, labels in loader:
            optimizer.zero_grad()
            outputs = model(features)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            seen_indices.extend(indices.tolist())


       print(
            f"\nrank={rank} | epoch={epoch} | "
            f"loss={epoch_loss:.4f} | "
            f"saw indices={seen_indices}"
        )



There is no explicit "average the gradients" call anywhere in this loop, and that is expected. The synchronization happens inside `loss.backward()`: because the model is wrapped in DDP, the backward pass averages each parameter's gradient across the processes as it runs. By the time `backward()` returns, every replica holds the same gradient, so `optimizer.step()` applies the same update on every process.

One small detail is worth naming. The call `sampler.set_epoch(epoch)` at the top of each epoch changes how the sampler shuffles, so the processes see a fresh split each epoch instead of repeating the same partition.

**Annotate it:**

1. Where in this loop do the gradients get averaged across processes, and what in the code tells you that?  
2. How many times per epoch do you expect the printed output to show up?  
3. Which chapter term does the averaging that happens inside `backward()` correspond to?

**What you should have found:**

1. The averaging happens inside `loss.backward()`, because the model is wrapped in DDP. There is no explicit synchronization line; the wrapper adds it to the backward pass.  
2. Every process prints its own message, so with a world size of 2 it will print the output line twice per epoch.  
3. It corresponds to gradient synchronization, carried out by the all-reduce collective communication described in the chapter.

### Step 6\. Saving the Model to Disk

DDP keeps the model parameters synchronized across ranks, but each process owns its own model instance. We only need to save one of them so that the trained model can later be reloaded for inference. The `if rank == 0` guard means only the first process saves. The guard is important because otherwise every process would attempt to save its own copy. If they all write to the same filename, those writes may conflict; if they use different filenames, you simply end up with unnecessary duplicate checkpoints.


In [ ]:
dist.barrier()

if rank == 0:
    torch.save(model.module.state_dict(), 'ddp_trained_model.pt')


Once again, a couple of small details are worth naming. The call `dist.barrier()` at the top ensures that all processes have reached the same point before proceeding any further. Even though it is not strictly required in our example, it can still be useful when later code depends on an output that’s produced by a single rank (e.g. reloading the saved model from disk). Additionally, the DDP model must be “unwrapped” to retrieve the underlying model that was trained. This is performed by referring to the `module` attribute of the DDP model.

**Annotate it:**

1. What would happen if you removed the `if rank == 0` guard?  
2. Would removing the `dist.barrier()` call before saving raise any issues?

**What you should have found:**

1. Every process would save its own model, so with a world size of 2 the saving operation would be executed twice per epoch. If they all use the same path, their writes could conflict. The `rank == 0` guard ensures that only one process writes the checkpoint.  
2. Normally, no. Rank 0's final backward pass has already synchronized its gradients with the other ranks, and its `optimizer.step()` has applied the final update. A barrier would make rank 0 wait until every other process reached the barrier, but it would not make rank 0's saved parameters more up to date.

### Step 7\. Cleaning Up and Launching the Processes

The last pieces shut the group down and start the processes in the first place.


In [ ]:
def cleanup():
    dist.destroy_process_group()

In [ ]:
def main():
    initial_model = make_model()
    dataset = make_dataset()
    num_epochs = 10
    world_size = 2
    mp.start_processes(train, args=(world_size, initial_model, dataset, num_epochs), nprocs=world_size, start_method='fork', join=True)



`destroy_process_group` releases the communication resources when training ends. Leaving it out can leave processes holding resources or waiting on one another, which is a common cause of a job that finishes its work but never exits.

`mp.start_processes` is what actually creates the processes. It launches `nprocs` copies of the `train` function, one per process, and passes each copy its own `rank` automatically (the first argument) along with the `world_size`, `initial_model`, `dataset`, `n_epochs` you supply in `args`. Here `world_size = 2`, so two CPU processes start, receive ranks 0 and 1, and run `train` together.

**Annotate it:**

1. What does `mp.start_processes(train, args=(world_size, initial_model, dataset, n_epochs), nprocs=world_size)` do, and where does each process's rank come from?  
2. What would happen if you removed the call to `cleanup` (and therefore `destroy_process_group`)?  
3. Which chapter term does the value `world_size = 2` set?

**What you should have found:**

1. It launches two copies of the `train` function as separate processes. Each process's rank is assigned automatically by `mp.start_processes` (0 and 1); the `world_size` and other arguments are passed explicitly through `args`.  
2. The training work would still complete, but the process group would not be torn down, so the processes could hang or hold resources instead of exiting cleanly.  
3. It sets the world size, the total number of processes in the job.

## (Optional) Run the Script Yourself

Running the script is not required. The exercise is complete once you have annotated the seven sections. If you want to watch the processes start and train, follow this section.

### Step 8\. Run

You can either save the complete listing from "The Complete Script" to a file named `ddp_cpu_demo.py` and then run it from a terminal using `python ddp_cpu_demo.py` or, if you’re using a notebook, run the last cell:


In [ ]:
if __name__ == "__main__":
    main()



Two CPU processes start, train for three epochs on their own slices of the synthetic data, and synchronize gradients each step. Each rank prints the epoch, its own loss, and the indices of the samples loaded by the sampler..

**Sample Output:**

```
rank=1 | epoch=0 | loss=52.3459 | saw indices=[137, 159, ... 347, 153]
rank=0 | epoch=0 | loss=45.3224 | saw indices=[172, 235, ... 134, 204]

rank=0 | epoch=1 | loss=1.1598 | saw indices=[37, 476, ... 387, 305]
rank=1 | epoch=1 | loss=1.1580 | saw indices=[158, 186, ,,, 85, 302]

rank=0 | epoch=2 | loss=0.1088 | saw indices=[168, 433, ... 342, 441]
rank=1 | epoch=2 | loss=0.1035 | saw indices=[179, 55, ... 2, 428]

rank=1 | epoch=3 | loss=0.0750 | saw indices=[165, 391, ... 142, 244]
rank=0 | epoch=3 | loss=0.0856 | saw indices=[362, 39, ... 170, 44]

rank=1 | epoch=4 | loss=0.0779 | saw indices=[511, 377, ... 272, 114]
rank=0 | epoch=4 | loss=0.0817 | saw indices=[122, 453, ... 289, 271]

rank=0 | epoch=5 | loss=0.0765 | saw indices=[355, 9, ... 58, 324]
rank=1 | epoch=5 | loss=0.0831 | saw indices=[125, 323, ... 43, 19]

rank=0 | epoch=6 | loss=0.0803 | saw indices=[394, 215, ,,, 419, 373]
rank=1 | epoch=6 | loss=0.0783 | saw indices=[113, 166, ... 262, 449]

rank=1 | epoch=7 | loss=0.0861 | saw indices=[203, 262, ... 510, 429]
rank=0 | epoch=7 | loss=0.0721 | saw indices=[175, 333, ... 15, 215]

rank=0 | epoch=8 | loss=0.0757 | saw indices=[451, 293, ... 469, 31]
rank=1 | epoch=8 | loss=0.0827 | saw indices=[62, 5, ... 510, 42]

rank=1 | epoch=9 | loss=0.0842 | saw indices=[356, 426, ... 162, 469]
rank=0 | epoch=9 | loss=0.0741 | saw indices=[382, 86, ... 157, 247]

```

The exact loss values will differ from run to run, because the data and the initial weights are random. What matters is that the run completes and exits cleanly, which together confirm that the process group was created, used, and destroyed as the script intends.

Finally, the trained model can be loaded from disk and its weights compared to the true weights used to generate the synthetic dataset used for training.


In [ ]:
# From the make_dataset() function
true_w = torch.tensor([[2.0], [-1.0], [0.5], [3.0]])
# Builds an untrained copy of the model
trained_model = make_model()
# Load trained weights
checkpoint = torch.load('ddp_trained_model.pt', map_location="cpu")
trained_model.load_state_dict(checkpoint)

print(f"Learned params: {trained_model.state_dict()['weight'].squeeze()}")
print(f"True params:    {true_w.squeeze()}")


**Sample Output:**

```
Learned params: tensor([ 1.9948, -0.9974,  0.4927,  2.9988])
True params:    tensor([ 2.0000, -1.0000,  0.5000,  3.0000])
```

The exact learned parameters may differ but, as long as they are close to the true parameters, the training was successful.

## Cleanup

The core reading-and-annotation exercise creates nothing and needs no cleanup.

If you completed the optional run, clean up as follows:

- The script calls `destroy_process_group` on its own before exiting, so the process group is already torn down when the run finishes. No manual teardown is needed.  
- If you ran it using the terminal, delete the file you saved if you do not want to keep it:


In [ ]:
rm ddp_cpu_demo.py


- If you ran it inside a managed environment such as Colab, restarting the runtime also clears everything.

## Summary

You read a complete DDP script and gave every part of it a name. The process group is created with `init_process_group` and named a backend (`"gloo"` on CPU); the `DistributedSampler` splits the data so each process trains on its own slice; `DDP(model)` wraps the model so gradients synchronize during the backward pass with no explicit code; and `destroy_process_group` shuts the group down at the end. You also saw that `mp.start_processes` launches the processes and assigns each one a rank, and that `world_size` is simply how many processes there are. The skill you practiced here is reading: opening an unfamiliar distributed script and recognizing its five structural pieces rather than skimming past them as boilerplate. That is the understanding the chapter set out to give you, and it is what lets you follow distributed training when it appears in the projects and code you work with.  
